# sales_forecast test notebook

Tests load_sales_csv, train_forecast_model, generate_forecast, and run_sales_forecast.

In [1]:
import pandas as pd
import numpy as np
from sales_forecast import load_sales_csv, train_forecast_model, generate_forecast, run_sales_forecast

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


## Create a synthetic monthly sales CSV for testing

In [2]:
dates = pd.date_range("2020-01-01", periods=48, freq="MS")
sales = 1000 + np.arange(48) * 20 + np.sin(np.arange(48) / 12 * 2 * np.pi) * 100
sample_df = pd.DataFrame({"date": dates.strftime("%Y-%m-%d"), "sales": sales.round(2)})
sample_df.to_csv("sample_sales.csv", index=False)
sample_df.head()

,date,sales
0,2020-01-01,1000.0
1,2020-02-01,1070.0
2,2020-03-01,1126.6
3,2020-04-01,1160.0
4,2020-05-01,1166.6


## Test load_sales_csv

In [3]:
df = load_sales_csv("sample_sales.csv")
assert list(df.columns) == ["ds", "y"]
assert len(df) == 48
assert df["ds"].is_monotonic_increasing
df.head()

,ds,y
0,2020-01-01,1000.0
1,2020-02-01,1070.0
2,2020-03-01,1126.6
3,2020-04-01,1160.0
4,2020-05-01,1166.6


## Test train_forecast_model

In [4]:
model = train_forecast_model(df)
print(type(model))

10:49:09 - cmdstanpy - INFO - Chain [1] start processing


10:49:09 - cmdstanpy - INFO - Chain [1] done processing


<class 'prophet.forecaster.Prophet'>


## Test generate_forecast

In [5]:
forecast = generate_forecast(model, periods=6, freq="MS")
assert list(forecast.columns) == ["ds", "yhat", "yhat_lower", "yhat_upper"]
assert len(forecast) == 48 + 6
forecast.tail(6)

,ds,yhat,yhat_lower,yhat_upper
48,2024-01-01,1959.998883,1957.699479,1961.987052
49,2024-02-01,2033.045829,2025.887291,2040.126421
50,2024-03-01,2087.624691,2073.507360,2101.039471
51,2024-04-01,2118.864163,2096.011599,2138.988587
52,2024-05-01,2133.551726,2100.672215,2162.664327
53,2024-06-01,2116.127240,2073.772010,2155.249168


## Test run_sales_forecast end to end

In [6]:
result = run_sales_forecast("sample_sales.csv", periods=30, freq="MS")
assert len(result) == 48 + 30
result.tail(5)

10:49:09 - cmdstanpy - INFO - Chain [1] start processing


10:49:10 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
73,2026-02-01,2518.838547,2143.555550,2924.162165
74,2026-03-01,2574.831850,2182.941233,3004.773921
75,2026-04-01,2607.560123,2199.730782,3068.761809
76,2026-05-01,2616.861927,2187.062774,3100.948542
77,2026-06-01,2599.935050,2151.805923,3104.786022


## Test error handling: missing file and wrong columns

In [7]:
try:
    load_sales_csv("does_not_exist.csv")
    raised = False
except FileNotFoundError:
    raised = True
assert raised

pd.DataFrame({"a": [1, 2], "b": [3, 4]}).to_csv("bad_cols.csv", index=False)
try:
    load_sales_csv("bad_cols.csv")
    raised = False
except ValueError:
    raised = True
assert raised

print("All error handling checks passed")

All error handling checks passed
